# Livrable Éthique — Projet IA for HumanForYou

**Formation :** PGE A4 FISA INFO — Intelligence Artificielle (Machine Learning)  
**Projet :** Analyse de l'attrition des employés — HumanForYou  
**Données :** HR Analytics — Kaggle (vjchoudhary7)


## Introduction

Ce document présente la démarche éthique adoptée tout au long du projet d'analyse de l'attrition des employés de l'entreprise pharmaceutique HumanForYou. Conformément aux **7 exigences recommandées par la Commission Européenne** pour une IA digne de confiance, nous avons questionné chacun de nos choix méthodologiques — de la sélection des données jusqu'aux recommandations finales — afin de garantir une approche responsable, transparente et respectueuse des individus.

L'objectif du projet est d'identifier les facteurs influençant le départ des employés et de proposer un modèle prédictif permettant à HumanForYou de mettre en place des actions de rétention ciblées. Cette finalité, bien qu'à visée organisationnelle, touche directement aux conditions de travail et à la vie professionnelle de personnes réelles, ce qui impose une **vigilance éthique particulière**.


## 1. Respect de l'autonomie humaine

### Principe
Le modèle d'IA ne doit pas se substituer à la décision humaine. Les employés concernés doivent rester acteurs de leur trajectoire professionnelle.

### Application au projet

**Ce que fait notre modèle :** Il attribue à chaque employé une probabilité de départ estimée sur la base de ses données professionnelles et personnelles. Il ne décide pas — il informe.

**Décision d'équipe :** Nous avons clairement positionné notre modèle comme un **outil d'aide à la décision RH**, et non comme un système de décision automatisé. Les recommandations produites doivent être soumises à l'interprétation et au jugement d'un responsable RH ou d'un manager avant toute action.

**Point de vigilance :** Le risque principal est que les résultats du modèle soient utilisés de manière automatique (ex : refus de promotion, surveillance accrue d'un employé identifié "à risque") sans intervention humaine. Nous recommandons à HumanForYou d'intégrer une charte d'utilisation précisant que toute décision individuelle doit être validée par un humain.

**Variables concernées :**
- `PerformanceRating`, `JobInvolvement` : issues de l'évaluation managériale — elles reflètent une appréciation subjective. Leur poids dans le modèle doit être interprété avec prudence pour éviter de pénaliser un employé sur la base d'un jugement partial.
- `avg_hours_worked` : calculée à partir des badgeuses — un employé parti tôt un jour pour raison médicale ne doit pas être systématiquement considéré comme "moins impliqué".


## 2. Robustesse technique et sécurité

### Principe
Le modèle doit être fiable, stable, et ses erreurs doivent être anticipées et minimisées.

### Application au projet

**Gestion du déséquilibre des classes :** Le jeu de données présente 84% d'employés restants contre 16% de départs. Sans correction, un modèle naïf prédirait systématiquement "reste" et afficherait 84% d'accuracy sans aucune valeur prédictive réelle. Nous avons utilisé le paramètre `class_weight='balanced'` pour corriger ce biais et prioriser la détection des vrais départs.

**Métriques choisies :** Nous avons privilégié l'AUC-ROC et le recall plutôt que l'accuracy seule, car dans ce contexte, **manquer un employé sur le départ (faux négatif) est plus coûteux** que de déclencher une action de rétention inutile (faux positif).

**Validation du modèle :** Un train/test split stratifié (80/20) a été appliqué pour évaluer les performances sur des données non vues. La stratification garantit que le ratio 84/16 est conservé dans les deux sous-ensembles, assurant une évaluation représentative.

**Point de vigilance :** Le modèle a été entraîné sur des données de 2015. Des changements organisationnels, économiques ou sectoriels depuis cette date peuvent rendre le modèle moins pertinent. Un **réentraînement périodique** est recommandé.

**Variables concernées :**
- `EnvironmentSatisfaction`, `JobSatisfaction`, `WorkLifeBalance` : issues d'une enquête **non obligatoire** — les valeurs manquantes ont été imputées par la médiane. Cette imputation introduit une approximation qui doit être signalée : les employés n'ayant pas répondu peuvent avoir des profils spécifiques (ex : désengagement).
- `avg_hours_worked` : calculée sur une année complète de badgeuse. Les jours sans données ont été exclus du calcul pour ne pas biaiser la moyenne.


## 3. Confidentialité et gouvernance des données

### Principe
Les données personnelles doivent être protégées, leur usage limité à la finalité annoncée, et leur traitement conforme aux réglementations en vigueur (RGPD).

### Application au projet

**Anonymisation :** Les données fournies par HumanForYou sont **anonymisées** — chaque employé est représenté uniquement par son `EmployeeID`, sans nom ni information directement identifiante. Nous avons supprimé cette colonne avant toute modélisation car elle n'a aucune valeur prédictive.

**Données sensibles identifiées et traitées :**

| Variable | Nature sensible | Décision d'équipe |
|----------|----------------|-------------------|
| `Gender` | Donnée à caractère personnel — sexe | Conservée pour analyse de biais, surveillée |
| `Age` | Donnée personnelle | Conservée (facteur légitime en RH) |
| `MaritalStatus` | Vie privée | Conservée avec vigilance (voir section 5) |
| `MonthlyIncome` | Donnée financière sensible | Conservée — facteur majeur d'attrition |

**Variables supprimées :**
- `EmployeeID` : identifiant direct — supprimé avant modélisation
- `EmployeeCount` : constante (valeur 1 pour tous) — aucune information prédictive
- `StandardHours` : constante (valeur 8 pour tous) — aucune information prédictive
- `Over18` : constante (True pour tous) — aucune information prédictive

### Inventaire complet des variables — Décisions d'inclusion/exclusion

Le principe de **minimisation des données** (RGPD, Article 5) impose de ne traiter que les données strictement nécessaires à l'objectif. Le tableau suivant documente, pour chaque variable du dataset, la décision d'inclusion ou d'exclusion et sa justification éthique et méthodologique.

| Variable | EDA | Modèle prédictif | Catégorie | Justification |
|----------|-----|-----------------|-----------|---------------|
| `EmployeeID` | ❌ | ❌ | Identifiant direct | Réidentification directe — supprimé avant toute analyse |
| `EmployeeCount` | ❌ | ❌ | Constante | Valeur identique (1) pour tous — aucune valeur prédictive |
| `StandardHours` | ❌ | ❌ | Constante | Valeur identique (8) pour tous — aucune variance exploitable |
| `Over18` | ❌ | ❌ | Constante | Valeur identique (True) pour tous — aucune information discriminante |
| `Gender` | ✅ | ❌ | Sensible | Conservée pour la détection de biais en EDA. **Exclue du modèle prédictif** : Son utilisation comme feature prédictive est illégale et discriminatoire. |
| `MaritalStatus` | ✅ | ❌ | Sensible | Conservée pour l'analyse bivariée uniquement. **Exclue du modèle prédictif** : cibler individuellement un employé sur la base de son statut marital serait discriminatoire et contraire à l'éthique RH. |
| `Age` | ✅ | ❌ | Démographique | Conservée pour l'analyse exploratoire (distribution par tranche d'âge). **Exclue du modèle prédictif** : bien que facteur légitime, son inclusion comme feature de scoring individuel présente un risque de discrimination par l'âge. |
| `MonthlyIncome` | ✅ | ✅ | Financier | Variable centrale et légitime. Levier d'action direct pour l'entreprise. |
| `JobSatisfaction` | ✅ | ✅ | Satisfaction | Prédicteur majeur de l'attrition. Valeurs manquantes imputées par la médiane. |
| `EnvironmentSatisfaction` | ✅ | ✅ | Satisfaction | Reflète la qualité perçue du cadre de travail. |
| `RelationshipSatisfaction` | ✅ | ✅ | Satisfaction | Indicateur des relations interpersonnelles au travail. |
| `WorkLifeBalance` | ✅ | ✅ | Satisfaction | Prédicteur du burn-out et de l'attrition, cohérent avec `avg_hours_worked`. |
| `JobInvolvement` | ✅ | ✅ | Évaluation managériale | Issue d'une appréciation subjective — interprétée avec prudence. |
| `PerformanceRating` | ✅ | ✅ | Évaluation managériale | Subjectivité identifiée. Ne doit pas pénaliser un employé sur un jugement partial. |
| `JobLevel` | ✅ | ✅ | Organisationnelle | Indicateur de position hiérarchique — facteur légitime. |
| `JobRole` | ✅ | ✅ | Organisationnelle | Identifie les postes à risque structurel (ex : Sales Executive). |
| `Department` | ✅ | ✅ | Organisationnelle | Permet l'analyse des disparités inter-départements. |
| `BusinessTravel` | ✅ | ✅ | Conditions de travail | Facteur de risque confirmé — fréquence de déplacement corrélée à l'attrition. |
| `DistanceFromHome` | ✅ | ✅ | Conditions de travail | Facteur de pénibilité légitime. Utilisé pour recommander des politiques de flexibilité. |
| `OverTime` | ✅ | ✅ | Conditions de travail | Indicateur de surcharge déclaré, complète `avg_hours_worked`. |
| `TotalWorkingYears` | ✅ | ✅ | Carrière | Indicateur d'expérience globale — facteur légitime. |
| `YearsAtCompany` | ✅ | ✅ | Carrière | Mesure la loyauté et l'ancrage dans l'entreprise. |
| `YearsInCurrentRole` | ✅ | ✅ | Carrière | Détecte les situations de stagnation de poste. |
| `YearsSinceLastPromotion` | ✅ | ✅ | Carrière | Signal fort d'absence de perspectives — facteur d'attrition majeur. |
| `YearsWithCurrManager` | ✅ | ✅ | Carrière | Reflète la stabilité et la qualité de la relation managériale. |
| `NumCompaniesWorked` | ✅ | ✅ | Carrière | Indicateur de mobilité naturelle — interprété comme profil, non comme défaut. |
| `TrainingTimesLastYear` | ✅ | ✅ | Développement | Investissement en formation corrélé négativement à l'attrition. |
| `PercentSalaryHike` | ✅ | ✅ | Financier | Indicateur de reconnaissance salariale récente. |
| `StockOptionLevel` | ✅ | ✅ | Financier | Levier de fidélisation financière à long terme. |
| `HourlyRate` / `DailyRate` / `MonthlyRate` | ✅ | ✅ | Financier | Compléments de rémunération. Corrélation avec `MonthlyIncome` vérifiée. |
| `Education` | ✅ | ✅ | Démographique | Niveau de formation — facteur légitime d'analyse de profil. |
| `EducationField` | ✅ | ✅ | Démographique | Domaine de formation — indique des dynamiques sectorielles. |
| `avg_hours_worked` | ✅ | ✅ | Construite (feature engineering) | Variable créée depuis les badgeuses (`in_time`, `out_time`), agrégée en moyenne annuelle. La donnée journalière brute n'est pas conservée. **Variable la plus prédictive du modèle.** |

**Note éthique :** `Gender`, `MaritalStatus` et `Age` ont été délibérément exclues du modèle prédictif malgré leur corrélation statistique avec l'attrition. Une corrélation statistique ne constitue pas une justification légale ou éthique suffisante pour intégrer ces critères dans un système de scoring individuel, leur usage contreviendrait au principe de non-discrimination. Ces variables ont néanmoins été conservées dans l'analyse exploratoire afin d'identifier d'éventuelles inégalités structurelles au sein de l'entreprise.

**Décision d'équipe :** Les données des badgeuses (`in_time`, `out_time`) constituent une **surveillance des horaires de travail**. Leur utilisation a été jugée légitime dans ce contexte car elles permettent de calculer une variable agrégée (`avg_hours_worked`) et non de tracer individuellement les employés heure par heure.

**Point de vigilance :** Dans un contexte réel, l'utilisation de données de badgeuse à des fins prédictives RH devrait faire l'objet d'une **information préalable des employés** et d'une consultation des représentants du personnel, conformément au RGPD (articles 13 et 14).


## 4. Transparence

### Principe
Le fonctionnement du modèle doit être explicable. Les parties prenantes doivent comprendre comment les décisions sont prises.

### Application au projet

**Choix du modèle :** Nous avons choisi la **régression logistique** comme modèle de référence, notamment pour son **interprétabilité** : les coefficients permettent de quantifier l'effet de chaque variable sur la probabilité de départ. Contrairement à des modèles de type boîte noire (deep learning), la régression logistique permet d'expliquer à un responsable RH pourquoi un employé est identifié comme à risque.

**Interprétation des coefficients :** Nous avons extrait et visualisé les 15 variables les plus influentes du modèle. Chaque variable est présentée avec son effet directionnel (augmente ou réduit le risque de départ).

**Documentation du notebook :** Chaque étape est documentée avec des cellules markdown expliquant les choix méthodologiques, les tests statistiques utilisés et leur interprétation.

**Point de vigilance :** Si un modèle plus complexe (Random Forest, XGBoost) est retenu pour ses meilleures performances, il faudra s'appuyer sur des outils d'explicabilité comme les **feature importances** pour maintenir la transparence vis-à-vis des utilisateurs finaux.


## 5. Diversité, non-discrimination et équité

### Principe
Le modèle ne doit pas reproduire ni amplifier des discriminations liées au genre, à l'âge, au statut marital ou à tout autre critère protégé.

### Application au projet

C'est l'exigence la plus critique pour ce projet car plusieurs variables du dataset sont des **critères potentiellement discriminatoires** au sens du droit du travail.

**`Gender` (Sexe)**  
- *Risque :* un modèle entraîné sur des données historiques peut reproduire des inégalités existantes (ex : si les femmes quittent davantage l'entreprise à cause d'un manque d'équité salariale, le modèle "apprend" que le genre prédit le départ).
- *Décision :* conservée pour l'analyse exploratoire afin d'identifier d'éventuelles inégalités, mais son utilisation comme critère de décision individuelle est exclue.
- *Point de vigilance :* **utiliser le genre comme critère de ciblage RH est illégal** en France (article L1132-1 du Code du travail).

**`Age` (Âge)**  
- *Risque :* discrimination par l'âge (seniors identifiés comme "moins fidèles").
- *Décision :* conservé comme facteur légitime d'analyse démographique. Les résultats doivent être présentés en termes de politiques de fidélisation par tranche d'âge, non de scoring individuel.

**`MaritalStatus` (Statut marital)**  
- *Risque :* les célibataires montrent un taux d'attrition plus élevé dans nos données. Cibler ces profils serait discriminatoire.
- *Décision :* conservé pour l'analyse bivariée uniquement. Recommandation explicite à HumanForYou de **ne jamais prendre de décision individuelle sur la base du statut marital**.

**`MonthlyIncome` (Salaire)**  
- *Risque faible* : variable légitime et centrale dans l'analyse de l'attrition.
- *Décision :* conservée sans restriction — c'est un levier d'action direct pour l'entreprise.

**Recommandation :** Vérifier lors du déploiement que les taux de faux positifs/négatifs sont comparables entre groupes (hommes/femmes, jeunes/seniors). Un écart significatif indiquerait un biais systémique à corriger.


## 6. Bien-être environnemental et sociétal

### Principe
Le développement et l'utilisation de l'IA doivent limiter leur impact environnemental et contribuer positivement à la société.

### Application au projet

**Impact environnemental :**  
Les modèles utilisés (régression logistique, Random Forest) sont des algorithmes classiques peu gourmands en ressources computationnelles. Leur empreinte carbone est négligeable comparée à l'entraînement de modèles de deep learning à grande échelle. Le projet a été développé localement, sans recours à des infrastructures cloud massives.

**Impact sociétal positif :**  
En aidant HumanForYou à réduire son taux d'attrition, le projet contribue à améliorer les **conditions de travail** des employés si les recommandations portent sur des leviers comme la satisfaction au travail, l'équilibre vie pro/perso ou les opportunités de formation. La réduction du turnover bénéficie également aux équipes en place : moins de surcharge liée aux départs, meilleure continuité des projets.

**Risque sociétal identifié :**  
Un usage mal encadré du modèle pourrait conduire à une **surveillance accrue** des employés ou à des pratiques de gestion par la peur. Cela aurait un effet délétère sur le bien-être au travail, à l'opposé de l'objectif initial.

**Décision d'équipe :** Nos recommandations finales se concentrent sur des **actions positives** (amélioration des conditions, formations, politique salariale) et non sur des mesures coercitives ou de surveillance individuelle.


## 7. Responsabilité

### Principe
Les responsabilités doivent être clairement définies. En cas d'erreur du modèle, des mécanismes de correction doivent exister.

### Application au projet

**Responsabilité de l'équipe projet :**  
Nous sommes responsables de la qualité de l'analyse, de la pertinence des modèles choisis et de la clarté des recommandations. Toute limitation identifiée (imputations, déséquilibre de classes, données datées de 2015) est documentée dans le notebook et dans ce livrable.

**Responsabilité de HumanForYou :**  
L'entreprise est responsable de l'usage qu'elle fait des résultats. Nous recommandons :
- Une **charte d'utilisation** précisant les usages autorisés et interdits
- Un **comité de revue** impliquant RH, managers et représentants des employés avant toute action basée sur les résultats
- Un **droit de recours** pour tout employé s'estimant lésé par une décision informée par le modèle

**Traçabilité :**  
Le notebook Jupyter constitue la trace complète et reproductible de notre démarche. Les choix d'exclusion de variables sont documentés et justifiés. Les hyperparamètres des modèles sont explicitement définis dans le code.

**Mécanismes de correction :**  
- Révision annuelle du modèle avec de nouvelles données
- Réévaluation régulière des métriques pour détecter une dérive (data drift)
- Processus de signalement des erreurs permettant aux RH de remonter les cas où le modèle s'est trompé


## Synthèse des points de vigilance

| Exigence | Point de vigilance principal | Action recommandée |
|----------|-----------------------------|-----------------|
| Autonomie humaine | Risque d'automatisation des décisions RH | Charte d'utilisation obligatoire |
| Robustesse | Données datées de 2015 | Réentraînement annuel |
| Confidentialité | Données de badgeuse et données sensibles | Information préalable des employés |
| Transparence | Modèles complexes peu interprétables | Privilégier les modèles explicables |
| Non-discrimination | Gender, Age, MaritalStatus dans le modèle | Ne jamais cibler individuellement sur ces critères |
| Bien-être sociétal | Risque de surveillance accrue | Recommandations orientées actions positives |
| Responsabilité | Absence de traçabilité des décisions | Comité de revue + droit de recours |

## Conclusion

La démarche éthique adoptée dans ce projet s'est construite progressivement, depuis le choix des variables jusqu'à la formulation des recommandations. Plusieurs décisions d'équipe ont été motivées par des préoccupations éthiques : la suppression des identifiants directs, la vigilance sur les variables potentiellement discriminatoires, le choix de modèles interprétables, et le positionnement systématique de notre outil comme aide à la décision humaine et non comme système autonome.

Le cas HumanForYou illustre une tension fondamentale de l'IA appliquée aux ressources humaines : les données les plus prédictives (genre, âge, statut marital) sont souvent celles qui présentent le plus grand risque éthique. La réponse à cette tension n'est pas de supprimer ces variables aveuglément, mais d'en encadrer strictement l'usage et de former les utilisateurs finaux à une lecture critique des résultats.


---
# Analyse ALTAI — Cadre de la Commission Européenne

Cette section complète la démarche éthique en appliquant formellement la **grille ALTAI** (Assessment List for Trustworthy AI) publiée par la Commission Européenne. Pour chaque exigence, les décisions d'équipe sont présentées sous forme de tableau questions/réponses/justifications.

**Contexte du projet :**

| Critère | Détail |
|--------|--------|
| Projet | Analyse et prédiction de l'attrition — HumanForYou |
| Données | Données RH personnelles (âge, salaire, satisfaction, horaires de badgeage, évaluations managers) |
| Contexte | Déploiement interne en entreprise — usage par les équipes RH |
| Algorithmes | EDA, clustering K-Means, tests statistiques (Mann-Whitney U, Chi²) |

Notre équipe s'est appuyée sur le cadre ALTAI pour structurer notre réflexion éthique tout au long du projet. Ce projet mobilise des données RH sensibles concernant des personnes identifiables via leur EmployeeID, ce qui en fait un système à fort enjeu éthique : les décisions ou recommandations issues du modèle peuvent directement affecter la carrière des employés.


### Exigence 1 — Respect de l'autonomie humaine et surveillance (ALTAI)

Notre système produit des analyses et des segmentations (clusters), pas des décisions automatiques. Il est conçu comme un outil d'aide à la décision pour les équipes RH, qui conservent l'entière responsabilité des actions engagées.

| Question ALTAI | Notre réponse | Justification |
|----------------|---------------|---------------|
| Le système prend-il des décisions automatiques affectant les employés ? | Non | Les sorties du modèle (clusters, variables significatives) sont présentées comme des indicateurs statistiques. Aucune décision de licenciement ou de sanction n'est automatisée. |
| Les utilisateurs RH sont-ils informés qu'ils travaillent avec un outil IA ? | Oui | Le livrable final précise explicitement la nature algorithmique des résultats (K-Means, tests statistiques). |
| Y a-t-il un risque de sur-dépendance des RH envers le modèle ? | Oui — risque identifié | Un manager pourrait stigmatiser un employé appartenant à un cluster à fort taux d'attrition sans chercher à comprendre son cas individuel. Ce risque a été discuté en équipe. |
| Des procédures anti-surreliance ont-elles été mises en place ? | Partiellement | Nous recommandons que les clusters soient utilisés comme point de départ d'un dialogue RH, jamais comme verdict. |
| Un mécanisme d'arrêt ou de correction est-il prévu ? | Oui | Les résultats du clustering peuvent être invalidés si le score de silhouette est trop faible. Le K optimal est réévalué à chaque mise à jour des données. |

### Exigence 2 — Robustesse technique et sécurité (ALTAI)

La robustesse du système repose sur la qualité des données en entrée, la stabilité du clustering, et la reproductibilité des analyses statistiques.

| Question ALTAI | Notre réponse | Justification |
|----------------|---------------|---------------|
| Des tests de robustesse ont-ils été effectués ? | Oui | Nous utilisons conjointement la méthode du coude (Elbow Method) et le score de silhouette pour valider le nombre de clusters K. Un seul critère serait insuffisant. |
| Les données sont-elles représentatives ? | Partiellement | Les données couvrent l'année 2015 uniquement. Une dérive temporelle est possible si les conditions de travail ont évolué depuis. |
| Un plan de secours est-il prévu en cas de données manquantes ? | Oui | Imputation par la médiane (variables numériques) et par le mode (variables catégorielles)  |
| La reproductibilité des résultats est-elle assurée ? | Oui | `random_state=42` est fixé sur tous les appels à KMeans, garantissant des résultats identiques à chaque exécution. |

### Exigence 3 — Confidentialité et gouvernance des données (ALTAI)

Les données traitées sont des données RH à caractère personnel : elles concernent des employés identifiables (via EmployeeID), incluent des informations sensibles (salaire, satisfaction, évaluations, horaires précis) et peuvent révéler des informations indirectes sur le bien-être des personnes.

| Question ALTAI | Notre réponse | Justification |
|----------------|---------------|---------------|
| Les données traitées sont-elles à caractère personnel ? | Oui | Toutes les données permettent d'identifier les individus et relèvent du RGPD. |
| L'EmployeeID est-il supprimé avant analyse ? | Oui | L'identifiant est supprimé du DataFrame principal après la phase de fusion. Cela limite le risque de réidentification dans les sorties. |
| Des mesures de minimisation des données sont-elles appliquées ? | Partiellement | Une revue des colonnes strictement nécessaires à l'objectif métier devrait être formalisée avec le DPO du client. |
| Le droit à l'effacement est-il pris en compte ? | À formaliser | Le notebook ne prévoit pas de mécanisme permettant de retirer un employé du dataset sur demande. Cette procédure doit être définie côté client. |
| Les données de badgeage sont-elles traitées avec précaution ? | Oui | Les fichiers in_time/out_time sont agrégés en une seule variable (`avg_hours_worked`) : la donnée brute journalière n'est pas conservée dans l'analyse finale. |

### Exigence 4 — Transparence (ALTAI)

La transparence est au cœur de notre démarche : chaque étape est documentée dans le notebook, chaque décision méthodologique est justifiée, et les limites des résultats sont explicitement mentionnées.

| Question ALTAI | Notre réponse | Justification |
|----------------|---------------|---------------|
| Les décisions du modèle sont-elles traçables ? | Oui | Le notebook documente l'ensemble du pipeline : chargement, fusion, nettoyage, feature engineering, analyse, clustering. Chaque cellule est commentée. |
| Le modèle est-il explicable pour des non-experts ? | Oui | Les résultats sont présentés sous forme de visualisations (boxplots, barplots, heatmap). Les p-values sont expliquées en langage courant dans une cellule dédiée. |
| Les limites sont-elles communiquées ? | Oui | La conclusion de l'EDA liste les points de vigilance pour la modélisation (déséquilibre, multicolinéarité, variable peu discriminante). |
| Les résultats du clustering sont-ils interprétables ? | Partiellement | Le profil moyen et le taux d'attrition par cluster sont calculés. Une description qualitative de chaque cluster devra être construite avec les RH pour être réellement actionnable. |


### Exigence 5 — Diversité, non-discrimination et équité (ALTAI)

Un système qui prédit le risque d'attrition peut, s'il n'est pas audité, reproduire des discriminations existantes. Si des employés d'un certain âge ou genre quittent davantage pour des raisons structurelles, le modèle apprendra ces patterns sans en questionner la légitimité.

| Question ALTAI | Notre réponse | Justification |
|----------------|---------------|---------------|
| Des tests de biais ont-ils été réalisés ? | Partiellement | L'analyse bivariée croise chaque variable (dont MaritalStatus, Gender, Age) avec l'attrition. Cela permet d'identifier des corrélations, mais pas d'établir si elles reflètent des discriminations structurelles. |
| Les groupes potentiellement discriminés ont-ils été identifiés ? | Oui | Célibataires, jeunes employés et grands voyageurs présentent des taux d'attrition plus élevés, ces résultats reflètent peut-être des conditions de travail défavorables, pas des prédispositions individuelles. |
| Une définition de l'équité a-t-elle été choisie ? | Non formalisée | Ce point n'a pas été explicitement traité dans le notebook. Il devra être discuté avec le client lors de la phase de modélisation. |
| Le dataset est-il représentatif de la diversité de l'entreprise ? | Inconnu | Nous ne disposons pas d'informations sur la composition réelle de l'effectif pour vérifier si certains groupes sont sous-représentés. |


### Exigence 6 — Bien-être environnemental et sociétal (ALTAI)

L'impact environnemental d'un projet d'analyse exploratoire sur des données RH de taille modérée est limité. L'impact sociétal, en revanche, est significatif : ce projet touche directement aux conditions de travail et à la relation employeur-employé.

| Question ALTAI | Notre réponse | Justification |
|----------------|---------------|---------------|
| L'impact environnemental a-t-il été évalué ? | Faible / Non critique | Le projet s'exécute sur des données de quelques milliers d'employés, sans entraînement de modèle profond. L'empreinte carbone est négligeable comparée à des projets de deep learning. |
| Le système pourrait-il affecter négativement le bien-être des employés ? | Oui, risque identifié | Si les résultats sont utilisés pour surveiller ou discriminer des employés à risque, l'impact sur le bien-être et la confiance serait négatif. |
| L'impact sur les compétences et l'emploi a-t-il été considéré ? | Oui | L'objectif est d'aider les RH à mieux cibler leurs actions de rétention, pas de supprimer des postes. Cette intention doit être formalisée dans les engagements contractuels. |
| Le système respecte-t-il les processus de dialogue social ? | À vérifier | L'utilisation de données de badgeage et d'évaluations pour segmenter les employés relève potentiellement du droit du travail français (consultation CSE). |


### Exigence 7 — Responsabilité (ALTAI)

La responsabilité implique que les acteurs impliqués (notre équipe, le client RH, les managers utilisateurs) sachent clairement qui est responsable de quoi, et que des mécanismes de recours existent si le système produit des effets négatifs sur un employé.

| Question ALTAI | Notre réponse | Justification |
|----------------|---------------|---------------|
| Les arbitrages méthodologiques ont-ils été documentés ? | Oui | Le choix de la médiane pour l'imputation, la suppression des colonnes non informatives, le choix du K optimal via double critère : toutes ces décisions sont tracées et justifiées dans le notebook. |
| Un mécanisme de signalement d'anomalie est-il prévu ? | Partiellement | Le notebook identifie les anomalies mais ne prévoit pas de procédure formelle pour les signaler au client en cours de production. |
| Des mécanismes de recours sont-ils prévus pour les employés affectés ? | Non, à mettre en place côté client | Si un employé fait l'objet d'une décision RH influencée par le modèle, il doit pouvoir demander une explication et contester le résultat. |
| Un audit externe est-il prévu ? | Recommandé | Nous recommandons qu'un audit éthique du modèle final soit réalisé par un tiers avant mise en production, pour vérifier l'absence de biais discriminatoires. |


### Synthèse des arbitrages et tensions identifiées (ALTAI)

| Tension | Exigences concernées | Arbitrage choisi | Justification |
|---------|----------------------|------------------|---------------|
| Utiliser `Age` et `Gender` améliore la précision mais crée un risque de discrimination | Robustesse (2) vs Équité (5) | Ces variables sont conservées dans l'EDA mais leur inclusion dans le modèle prédictif devra être encadrée juridiquement | Le compromis entre performance et équité nécessite un arbitrage RH et juridique, pas uniquement data |
| La granularité des données de badgeage améliore la feature mais constitue une surveillance fine | Robustesse (2) vs Confidentialité (3) | Les données brutes horaires sont agrégées en une variable moyenne par employé. La donnée journalière détaillée n'est pas conservée | La réduction de granularité préserve la vie privée tout en conservant l'information utile (charge de travail moyenne) |
| Communiquer les clusters aux managers améliore l'action RH mais risque de stigmatiser des employés | Transparence (4) vs Bien-être (6) | Les clusters sont présentés comme des profils statistiques avec mise en garde contre toute utilisation normative ou discriminante | La transparence ne doit pas se faire au détriment du bien-être, la contextualisation des résultats est impérative |

### Références

- Commission Européenne — Assessment List for Trustworthy AI (ALTAI), juillet 2020.
- AI HLEG — Ethics Guidelines for Trustworthy AI, avril 2019.
- RGPD — Article 22 : Droit à ne pas faire l'objet d'une décision automatisée.
- Code du travail français — Article L.2312-38 : Consultation du CSE sur les outils algorithmiques.
- CNIL — Fiche pratique : IA et ressources humaines.
